In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)

device = ("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
transform = transforms.ToTensor()

full_train = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
official_test = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_subset, val_subset = random_split(
    full_train, [48_000, 12_000],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

X, y = next(iter(train_loader))
print("Images:", X.shape, "Labels:", y.shape)
print("Train:", len(train_subset), "Validation:", len(val_subset))

Images: torch.Size([64, 1, 28, 28]) Labels: torch.Size([64])
Train: 48000 Validation: 12000


In [3]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.network(x)

In [4]:
model = FashionMLP().to(device)
print(model)

logits = model(X.to(device))
print("Logits:", logits.shape)

FashionMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=64, bias=True)
    (6): ReLU()
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)
Logits: torch.Size([64, 10])


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size

    return total_loss / total, correct / total

In [7]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += batch_size

    return total_loss / total, correct / total

In [8]:
history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch + 1:02d} | train loss={train_loss:.4f}, acc={train_acc:.2%} | val loss={val_loss:.4f}, acc={val_acc:.2%}")

Epoch 01 | train loss=0.6044, acc=78.17% | val loss=0.4287, acc=84.79%
Epoch 02 | train loss=0.4005, acc=85.55% | val loss=0.3923, acc=85.38%
Epoch 03 | train loss=0.3536, acc=87.04% | val loss=0.3683, acc=87.02%
Epoch 04 | train loss=0.3259, acc=87.90% | val loss=0.3300, acc=88.05%
Epoch 05 | train loss=0.3096, acc=88.60% | val loss=0.3428, acc=87.93%


In [9]:
train_targets = full_train.targets[train_subset.indices]
class_counts = torch.bincount(train_targets, minlength=10)
majority_class = class_counts.argmax().item()

val_targets = full_train.targets[val_subset.indices]
baseline_accuracy = (val_targets == majority_class).float().mean().item()

print("Majority class:", majority_class)
print(f"Majority baseline: {baseline_accuracy:.2%}")
print(f"MLP validation accuracy: {history['val_accuracy'][-1]:.2%}")
assert history["val_accuracy"][-1] > baseline_accuracy, "Model не превзошла baseline"
assert history["train_loss"][-1] < history["train_loss"][0], "Train loss не снизился"

Majority class: 7
Majority baseline: 9.45%
MLP validation accuracy: 87.93%


In [10]:
weights_path = "fashion_mnist_mlp.pth"
torch.save(model.state_dict(), weights_path)

loaded_model = FashionMLP().to(device)
loaded_model.load_state_dict(
    torch.load(weights_path, map_location=device, weights_only=True)
)

def predict_classes(model, loader):
    model.eval()
    predictions = []

    with torch.no_grad():
        for X, _ in loader:
            logits = model(X.to(device))
            predictions.append(logits.argmax(dim=1).cpu())

    return torch.cat(predictions)

original_predictions = predict_classes(model, val_loader)
loaded_predictions = predict_classes(loaded_model, val_loader)
print("Done")

assert torch.equal(original_predictions, loaded_predictions), \
    "Predictions differ after loading!"

print("Predictions match!")

Done
Predictions match!
